# 🎮 Task 2: Player Segment Classification (CPE 342 - Team ML009)

- **Objective:** Multi-class classification (`0=Casual, 1=Grinder, 2=Social, 3=Whale`)
- **Evaluation Metric:** Macro F1 Score
- **Pipeline:** EDA → Feature Engineering → 5-Fold Stratified CV (LightGBM, XGBoost, CatBoost) → Weighted Probability Ensemble → Export `submissions/sub_task2.csv`

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
N_SPLITS = 5
NUM_CLASSES = 4
CLASS_NAMES = {0: "Casual", 1: "Grinder", 2: "Social", 3: "Whale"}

np.random.seed(SEED)

## 1. Load Data & Exploratory Data Analysis (EDA)

In [ ]:
# Auto-detect data directory
CANDIDATE_DIRS = ["../data/task2", "data/task2", "../CPE342-Hackathon/task2"]
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(d)), "../data/task2")
print(f"Using DATA_DIR: {os.path.abspath(DATA_DIR)}")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# Load datacard (supports datacard.csv or datacard.txt)
for dc_name in ["datacard.csv", "datacard.txt"]:
    dc_path = os.path.join(DATA_DIR, dc_name)
    if os.path.exists(dc_path):
        print(f"\n--- {dc_name} ---")
        if dc_name.endswith(".csv"):
            display(pd.read_csv(dc_path))
        else:
            with open(dc_path, "r", encoding="utf-8") as f:
                print(f.read()[:2000])
        break

print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

In [ ]:
# Identify ID and Target columns
ID_COL = "id" if "id" in train_df.columns else train_df.columns[0]
TARGET_COL = "segment" if "segment" in train_df.columns else [c for c in train_df.columns if c not in test_df.columns][0]
print(f"ID_COL: {ID_COL} | TARGET_COL: {TARGET_COL}")

# Check Missing Values
missing_train = train_df.isna().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print("\nTop Missing Values in Train:")
print(missing_train.head(15) if len(missing_train) > 0 else "No missing values!")

# Check Class Distribution
class_counts = train_df[TARGET_COL].value_counts().sort_index()
print("\nClass Distribution:")
for k, v in class_counts.items():
    print(f"  Class {k} ({CLASS_NAMES.get(k, k)}): {v} ({v / len(train_df) * 100:.2f}%)")

plt.figure(figsize=(6, 4))
sns.barplot(x=[f"{k}: {CLASS_NAMES.get(k, k)}" for k in class_counts.index], y=class_counts.values, palette="viridis")
plt.title("Task 2 - Target Class Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 2. Feature Engineering
สร้าง Ratio & Interaction features เพื่อแยกพฤติกรรมสายเปย์ (`Whale`), สายฟาร์ม (`Grinder`), สายเพื่อนเยอะ (`Social`), และผู้เล่นทั่วไป (`Casual`)

In [ ]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create domain-specific ratio and interaction features for Player Segment Classification."""
    df = df.copy()
    cols = set(df.columns)

    # 1. Total weekly playtime = play_frequency * avg_session_duration
    if {"play_frequency", "avg_session_duration"}.issubset(cols):
        df["total_playtime_week"] = df["play_frequency"] * df["avg_session_duration"]

    # 2. Spending per playtime/session
    if {"avg_monthly_spending", "total_playtime_week"}.issubset(set(df.columns)):
        df["spend_per_playtime"] = df["avg_monthly_spending"] / (df["total_playtime_week"] + 1.0)

    # 3. Social intensity (friends per playtime)
    if {"friend_count", "total_playtime_week"}.issubset(set(df.columns)):
        df["social_intensity"] = df["friend_count"] / (df["total_playtime_week"] + 1.0)

    # 4. Monthly spending ratio to lifetime total spending
    if {"avg_monthly_spending", "total_spending_thb"}.issubset(cols):
        df["monthly_to_total_spend_ratio"] = df["avg_monthly_spending"] / (df["total_spending_thb"] + 1.0)

    # 5. Row-wise missing count (can carry signal)
    df["missing_count"] = df.isna().sum(axis=1)

    return df


train_fe = create_features(train_df)
test_fe = create_features(test_df)

feature_cols = [c for c in train_fe.columns if c not in [ID_COL, TARGET_COL]]

# Encode categorical columns strictly using train set only (no test set leakage)
cat_cols = train_fe[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    trn_vals = train_fe[col].fillna("MISSING").astype(str)
    le.fit(trn_vals)
    mapping = {label: idx for idx, label in enumerate(le.classes_)}
    train_fe[col] = trn_vals.map(mapping).fillna(-1).astype(int)
    test_fe[col] = test_fe[col].fillna("MISSING").astype(str).map(mapping).fillna(-1).astype(int)

X = train_fe[feature_cols].copy()
y = train_fe[TARGET_COL].astype(int).values
X_test = test_fe[feature_cols].copy()

print(f"Total features: {len(feature_cols)} (categorical encoded: {len(cat_cols)})")

## 3. 5-Fold Stratified CV Training (LightGBM, XGBoost, CatBoost)

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros((len(X), NUM_CLASSES))
oof_xgb = np.zeros((len(X), NUM_CLASSES))
oof_cat = np.zeros((len(X), NUM_CLASSES))

test_lgb = np.zeros((len(X_test), NUM_CLASSES))
test_xgb = np.zeros((len(X_test), NUM_CLASSES))
test_cat = np.zeros((len(X_test), NUM_CLASSES))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_trn, y_trn = X.iloc[trn_idx], y[trn_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]

    # --- 1. LightGBM ---
    model_lgb = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=NUM_CLASSES,
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=31,
        subsample_for_bin=50000,
        colsample_bytree=0.8,
        subsample=0.8,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1,
    )
    model_lgb.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)
    test_lgb += model_lgb.predict_proba(X_test) / N_SPLITS

    # --- 2. XGBoost ---
    model_xgb = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        eval_metric="mlogloss",
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        early_stopping_rounds=50,
        random_state=SEED + fold,
        n_jobs=-1,
    )
    model_xgb.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)
    test_xgb += model_xgb.predict_proba(X_test) / N_SPLITS

    # --- 3. CatBoost ---
    model_cat = CatBoostClassifier(
        loss_function="MultiClass",
        classes_count=NUM_CLASSES,
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        random_seed=SEED + fold,
        verbose=False,
    )
    model_cat.fit(
        X_trn, y_trn,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        verbose=False,
    )
    oof_cat[val_idx] = model_cat.predict_proba(X_val)
    test_cat += model_cat.predict_proba(X_test) / N_SPLITS

    f1_l = f1_score(y_val, oof_lgb[val_idx].argmax(axis=1), average="macro")
    f1_x = f1_score(y_val, oof_xgb[val_idx].argmax(axis=1), average="macro")
    f1_c = f1_score(y_val, oof_cat[val_idx].argmax(axis=1), average="macro")
    print(f"Fold {fold} | Macro F1 -> LGB: {f1_l:.5f} | XGB: {f1_x:.5f} | CAT: {f1_c:.5f}")

print("\n--- Overall Out-of-Fold (OOF) Macro F1 ---")
print(f"LightGBM OOF Macro F1 : {f1_score(y, oof_lgb.argmax(axis=1), average='macro'):.5f}")
print(f"XGBoost  OOF Macro F1 : {f1_score(y, oof_xgb.argmax(axis=1), average='macro'):.5f}")
print(f"CatBoost OOF Macro F1 : {f1_score(y, oof_cat.argmax(axis=1), average='macro'):.5f}")

## 4. Ensembling (Weighted Average Probabilities) & Evaluation

In [ ]:
# Grid search optimal blend weights for LightGBM, XGBoost, CatBoost
best_f1 = -1.0
best_weights = (1/3, 1/3, 1/3)

for w1 in np.linspace(0, 1, 11):
    for w2 in np.linspace(0, 1 - w1, int(round((1 - w1) * 10)) + 1):
        w3 = max(0.0, 1.0 - w1 - w2)
        blend_oof = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        score = f1_score(y, blend_oof.argmax(axis=1), average="macro")
        if score > best_f1:
            best_f1 = score
            best_weights = (w1, w2, w3)

w1, w2, w3 = best_weights
oof_ens = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
test_ens = w1 * test_lgb + w2 * test_xgb + w3 * test_cat

print(f"Best Ensemble Weights (LGB, XGB, CAT): ({w1:.2f}, {w2:.2f}, {w3:.2f})")
print(f"Ensemble OOF Macro F1: {best_f1:.5f}\n")
print(classification_report(y, oof_ens.argmax(axis=1), target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)]))

# Confusion Matrix
cm = confusion_matrix(y, oof_ens.argmax(axis=1))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
            yticklabels=[CLASS_NAMES[i] for i in range(NUM_CLASSES)])
plt.title("Ensemble OOF Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 5. Save Task 2 Predictions (`submissions/sub_task2.csv`)

In [ ]:
os.makedirs("../submissions", exist_ok=True)
sub_path = "../submissions/sub_task2.csv"

sub_df = pd.DataFrame({
    "id": test_df[ID_COL].values,
    "task2": test_ens.argmax(axis=1).astype(int),
})

assert not sub_df.isna().any().any(), "Found NaN in predictions!"
assert set(sub_df["task2"].unique()).issubset({0, 1, 2, 3}), "Invalid class labels!"

sub_df.to_csv(sub_path, index=False)
print(f"✅ Saved Task 2 predictions to {os.path.abspath(sub_path)} ({len(sub_df)} rows)")
print("Predicted class distribution:")
print(sub_df["task2"].value_counts().sort_index())